<a href="https://colab.research.google.com/github/SaiLinHtet-KiRA/Big-Data-Analytics/blob/main/Week_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part A — Google Colab Setup and SparkSession**

## *Step A2 — Install PySpark*

In [138]:
!pip install -q pyspark

## **Step A3 — Create the SparkSession**

In [139]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CSC475Week7Lab")
    .getOrCreate()
)

print("Spark version:", spark.version)


Spark version: 4.0.4


## **Part B — Create the Dataset and Inspect Partitions**

In [140]:
sales_data = [
    ("Bangkok", "Electronics", "Laptop", 35000),
    ("Bangkok", "Accessories", "Mouse", 500),
    ("ChiangMai", "Electronics", "Laptop", 34000),
    ("Bangkok", "Accessories", "Keyboard", 1500),
    ("Phuket", "Food", "Coffee", 120),
    ("ChiangMai", "Food", "Coffee", 100),
]

columns = ["city", "category", "product", "amount"]


## **Step B2 — Inspect partitioning with a simple numeric RDD**

In [141]:
partition_demo = spark.sparkContext.parallelize(range(20), 4)

print("Number of partitions:", partition_demo.getNumPartitions())
print(partition_demo.glom().collect())


Number of partitions: 4
[[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14], [15, 16, 17, 18, 19]]


# **Part C — RDD Programming: Transformation, Action, and Aggregation**

In [142]:
sales_rdd = spark.sparkContext.parallelize(
    [(row[0], row[3]) for row in sales_data],
    3
)

print("RDD partitions:", sales_rdd.getNumPartitions())
print(sales_rdd.collect())


RDD partitions: 3
[('Bangkok', 35000), ('Bangkok', 500), ('ChiangMai', 34000), ('Bangkok', 1500), ('Phuket', 120), ('ChiangMai', 100)]


## **Step C2 — Apply map() as a transformation**

In [143]:
with_tax_rdd = sales_rdd.map(
    lambda x: (x[0], round(x[1] * 1.07, 2))
)

# No action has been called yet.
print(type(with_tax_rdd))


<class 'pyspark.core.rdd.PipelinedRDD'>


## **Step C3 — Trigger execution with collect()**

In [144]:
print(with_tax_rdd.collect())

[('Bangkok', 37450.0), ('Bangkok', 535.0), ('ChiangMai', 36380.0), ('Bangkok', 1605.0), ('Phuket', 128.4), ('ChiangMai', 107.0)]


## **Step C4 — Filter high-value transactions**

In [145]:
high_value_rdd = sales_rdd.filter(
    lambda x: x[1] > 1000
)

print(high_value_rdd.collect())


[('Bangkok', 35000), ('ChiangMai', 34000), ('Bangkok', 1500)]


## **Step C5 — Aggregate with reduceByKey()**

In [146]:
city_totals_rdd = (
    sales_rdd
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
)

print(city_totals_rdd.collect())


[('Bangkok', 37000), ('ChiangMai', 34100), ('Phuket', 120)]


# **Part D — Observe Lazy Evaluation**

## **Step D1 — Define several transformations**

In [147]:
filtered_rdd = sales_rdd.filter(
    lambda x: x[1] > 1000
)

adjusted_rdd = filtered_rdd.map(
    lambda x: (x[0], round(x[1] * 1.07, 2))
)

print("Transformations have been defined.")


Transformations have been defined.


## **Step D2 — Call an action**

In [148]:
print(adjusted_rdd.collect())

[('Bangkok', 37450.0), ('ChiangMai', 36380.0), ('Bangkok', 1605.0)]


## **Part E — Spark DataFrame Fundamentals**

In [149]:
df = spark.createDataFrame(
    sales_data,
    columns
)


## **Step E2 — Preview records and schema**

In [150]:
df.show(truncate=False)
df.printSchema()
print("Columns:", df.columns)
print("Row count:", df.count())


+---------+-----------+--------+------+
|city     |category   |product |amount|
+---------+-----------+--------+------+
|Bangkok  |Electronics|Laptop  |35000 |
|Bangkok  |Accessories|Mouse   |500   |
|ChiangMai|Electronics|Laptop  |34000 |
|Bangkok  |Accessories|Keyboard|1500  |
|Phuket   |Food       |Coffee  |120   |
|ChiangMai|Food       |Coffee  |100   |
+---------+-----------+--------+------+

root
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)

Columns: ['city', 'category', 'product', 'amount']
Row count: 6


## **Step E3 — Check the DataFrame partition count**

In [151]:
print("DataFrame partitions:", df.rdd.getNumPartitions())

DataFrame partitions: 2


# **Part F — DataFrame Transformations**

## **Step F1 — Filter rows**

In [152]:
high_value_df = df.filter(
    F.col("amount") > 1000
)

high_value_df.orderBy("city", "amount").show(truncate=False)


+---------+-----------+--------+------+
|city     |category   |product |amount|
+---------+-----------+--------+------+
|Bangkok  |Accessories|Keyboard|1500  |
|Bangkok  |Electronics|Laptop  |35000 |
|ChiangMai|Electronics|Laptop  |34000 |
+---------+-----------+--------+------+



## **Step F2 — Select only the columns needed for analysis**

In [153]:
selected_df = df.select(
    "city", "product", "amount"
)

selected_df.show(truncate=False)


+---------+--------+------+
|city     |product |amount|
+---------+--------+------+
|Bangkok  |Laptop  |35000 |
|Bangkok  |Mouse   |500   |
|ChiangMai|Laptop  |34000 |
|Bangkok  |Keyboard|1500  |
|Phuket   |Coffee  |120   |
|ChiangMai|Coffee  |100   |
+---------+--------+------+



## **Step F3 — Create a derived column**

In [154]:
df_with_tax = df.withColumn(
    "amount_with_tax",
    F.round(F.col("amount") * 1.07, 2)
)

df_with_tax.show(truncate=False)


+---------+-----------+--------+------+---------------+
|city     |category   |product |amount|amount_with_tax|
+---------+-----------+--------+------+---------------+
|Bangkok  |Electronics|Laptop  |35000 |37450.0        |
|Bangkok  |Accessories|Mouse   |500   |535.0          |
|ChiangMai|Electronics|Laptop  |34000 |36380.0        |
|Bangkok  |Accessories|Keyboard|1500  |1605.0         |
|Phuket   |Food       |Coffee  |120   |128.4          |
|ChiangMai|Food       |Coffee  |100   |107.0          |
+---------+-----------+--------+------+---------------+



# **Part G — Grouping and Aggregation with DataFrames**

## **Step G1 — Total sales by city**

In [155]:
city_totals_df = (
    df.groupBy("city")
      .agg(F.sum("amount").alias("total_sales"))
      .orderBy(F.desc("total_sales"))
)

city_totals_df.show()


+---------+-----------+
|     city|total_sales|
+---------+-----------+
|  Bangkok|      37000|
|ChiangMai|      34100|
|   Phuket|        120|
+---------+-----------+



## **Step G2 — Multiple aggregations by category**

In [156]:
category_summary = (
    df.groupBy("category")
      .agg(
          F.sum("amount").alias("total_sales"),
          F.round(F.avg("amount"), 2).alias("avg_sales"),
          F.count("*").alias("transactions"),
          F.max("amount").alias("max_sale")
      )
      .orderBy(F.desc("total_sales"))
)

category_summary.show()


+-----------+-----------+---------+------------+--------+
|   category|total_sales|avg_sales|transactions|max_sale|
+-----------+-----------+---------+------------+--------+
|Electronics|      69000|  34500.0|           2|   35000|
|Accessories|       2000|   1000.0|           2|    1500|
|       Food|        220|    110.0|           2|     120|
+-----------+-----------+---------+------------+--------+



## **Step G3 — Inspect an execution plan**

In [157]:
pipeline = (
    df.filter(F.col("amount") > 1000)
      .groupBy("city")
      .agg(F.sum("amount").alias("total_high_value_sales"))
)

pipeline.explain()
pipeline.orderBy(F.desc("total_high_value_sales")).show()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#1127], functions=[sum(amount#1130L)])
   +- Exchange hashpartitioning(city#1127, 200), ENSURE_REQUIREMENTS, [plan_id=3011]
      +- HashAggregate(keys=[city#1127], functions=[partial_sum(amount#1130L)])
         +- Project [city#1127, amount#1130L]
            +- Filter (isnotnull(amount#1130L) AND (amount#1130L > 1000))
               +- Scan ExistingRDD[city#1127,category#1128,product#1129,amount#1130L]


+---------+----------------------+
|     city|total_high_value_sales|
+---------+----------------------+
|  Bangkok|                 36500|
|ChiangMai|                 34000|
+---------+----------------------+



## **Part H — Compare RDD and DataFrame Results**

In [158]:
rdd_result = dict(city_totals_rdd.collect())

df_result = {
    row["city"]: row["total_sales"]
    for row in city_totals_df.collect()
}

print("RDD result:", rdd_result)
print("DataFrame result:", df_result)
print("Same totals?", rdd_result == df_result)


RDD result: {'Bangkok': 37000, 'ChiangMai': 34100, 'Phuket': 120}
DataFrame result: {'Bangkok': 37000, 'ChiangMai': 34100, 'Phuket': 120}
Same totals? True


# **Part I — Guided Coding Activity**

## **Question 1 — Display the schema and all records**

In [159]:
df.printSchema()
df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)

+---------+-----------+--------+------+
|city     |category   |product |amount|
+---------+-----------+--------+------+
|Bangkok  |Electronics|Laptop  |35000 |
|Bangkok  |Accessories|Mouse   |500   |
|ChiangMai|Electronics|Laptop  |34000 |
|Bangkok  |Accessories|Keyboard|1500  |
|Phuket   |Food       |Coffee  |120   |
|ChiangMai|Food       |Coffee  |100   |
+---------+-----------+--------+------+



## **Question 2 — Filter transactions with amount greater than 1,000**

In [160]:
df.filter(F.col("amount") > 1000).show(truncate=False)

+---------+-----------+--------+------+
|city     |category   |product |amount|
+---------+-----------+--------+------+
|Bangkok  |Electronics|Laptop  |35000 |
|ChiangMai|Electronics|Laptop  |34000 |
|Bangkok  |Accessories|Keyboard|1500  |
+---------+-----------+--------+------+



## **Question 3 — Calculate total sales by city**

In [161]:
df.groupBy("city").agg(
    F.sum("amount").alias("total_sales")
).show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|ChiangMai|      34100|
|  Bangkok|      37000|
|   Phuket|        120|
+---------+-----------+



## **Question 4 — Calculate total sales by category**

In [162]:
df.groupBy("category").agg(
    F.sum("amount").alias("total_sales")
).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|      69000|
|Accessories|       2000|
|       Food|        220|
+-----------+-----------+



## **Question 5 — Calculate average transaction amount by city**

In [163]:
df.groupBy("city").agg(
    F.round(F.avg("amount"), 2).alias("avg_amount")
).show()

+---------+----------+
|     city|avg_amount|
+---------+----------+
|ChiangMai|   17050.0|
|  Bangkok|  12333.33|
|   Phuket|     120.0|
+---------+----------+



## **Question 6 — Count transactions by city**

In [164]:
df.groupBy("city").agg(
    F.count("*").alias("transactions")
).show()

+---------+------------+
|     city|transactions|
+---------+------------+
|ChiangMai|           2|
|  Bangkok|           3|
|   Phuket|           1|
+---------+------------+



## **Question 7 — Sort city summary from highest to lowest total sales**

In [165]:
city_summary = (
    df.groupBy("city")
      .agg(
          F.sum("amount").alias("total_sales")
      )
      .orderBy(F.desc("total_sales"))
)

city_summary.show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|  Bangkok|      37000|
|ChiangMai|      34100|
|   Phuket|        120|
+---------+-----------+



## **Question 8 — Write two sentences interpreting the result**

Bangkok has the highest total sales at 37,000, followed by ChiangMai at 34,100. Phuket has the lowest total sales at 120, showing that Bangkok and ChiangMai contribute most of the sales in this dataset.

## **Question 9 — Classify the main operations as transformations or actions**

# **Mini Coding Challenge**

## **Challenge A — Count transactions by city**

In [166]:
df.groupBy("city").count().orderBy(F.desc("count")).show()

+---------+-----+
|     city|count|
+---------+-----+
|  Bangkok|    3|
|ChiangMai|    2|
|   Phuket|    1|
+---------+-----+



## **Challenge B — Maximum transaction amount by category**

In [167]:
df.groupBy("category") \
  .agg(F.max("amount").alias("max_amount")) \
  .orderBy("category") \
  .show()


+-----------+----------+
|   category|max_amount|
+-----------+----------+
|Accessories|      1500|
|Electronics|     35000|
|       Food|       120|
+-----------+----------+



## **Challenge C — Electronics sales by city**

In [168]:
(df.filter(F.col("category") == "Electronics")
   .groupBy("city")
   .agg(F.sum("amount").alias("electronics_sales"))
   .orderBy(F.desc("electronics_sales"))
   .show())


+---------+-----------------+
|     city|electronics_sales|
+---------+-----------------+
|  Bangkok|            35000|
|ChiangMai|            34000|
+---------+-----------------+



## **Challenge D — Add a 7% tax-adjusted amount**

In [169]:
df.withColumn(
    "amount_with_tax",
    F.round(F.col("amount") * 1.07, 2)
).show(truncate=False)


+---------+-----------+--------+------+---------------+
|city     |category   |product |amount|amount_with_tax|
+---------+-----------+--------+------+---------------+
|Bangkok  |Electronics|Laptop  |35000 |37450.0        |
|Bangkok  |Accessories|Mouse   |500   |535.0          |
|ChiangMai|Electronics|Laptop  |34000 |36380.0        |
|Bangkok  |Accessories|Keyboard|1500  |1605.0         |
|Phuket   |Food       |Coffee  |120   |128.4          |
|ChiangMai|Food       |Coffee  |100   |107.0          |
+---------+-----------+--------+------+---------------+



# **Part J — Optional Extension: Read the CSV File**

## **Step J1 — Upload sales_week7_lab.csv to Colab**

In [170]:
from google.colab import files

uploaded = files.upload()


Saving sales_week7_lab.csv to sales_week7_lab (1).csv


## **Step J2 — Read the CSV with Spark**

In [171]:
csv_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("sales_week7_lab.csv")
)

csv_df.show()
csv_df.printSchema()


+---------+-----------+--------+------+
|     city|   category| product|amount|
+---------+-----------+--------+------+
|  Bangkok|Electronics|  Laptop| 35000|
|  Bangkok|Accessories|   Mouse|   500|
|ChiangMai|Electronics|  Laptop| 34000|
|  Bangkok|Accessories|Keyboard|  1500|
|   Phuket|       Food|  Coffee|   120|
|ChiangMai|       Food|  Coffee|   100|
+---------+-----------+--------+------+

root
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)



# **Part K — Lab Review and Submission Checklist**

## **Question 1 — What is the role of SparkSession?**

In [172]:
print("SparkSession:", spark)
print("Spark version:", spark.version)

SparkSession: <pyspark.sql.session.SparkSession object at 0x7ce716df1be0>
Spark version: 4.0.4


## **Question 2 — What is a partition, and why does it matter?**

In [173]:
print("CSV DataFrame partitions:", csv_df.rdd.getNumPartitions())

partition_demo = spark.sparkContext.parallelize(range(20), 4)

print("Demo partitions:", partition_demo.getNumPartitions())
print("Data in each partition:", partition_demo.glom().collect())

CSV DataFrame partitions: 1
Demo partitions: 4
Data in each partition: [[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14], [15, 16, 17, 18, 19]]


## **Question 3 — What is the difference between a transformation and an action?**

In [174]:
filtered_df = csv_df.filter(F.col("amount") > 1000)  # Transformation

print("Transformation created:")
filtered_df.explain()

print("Action result:")
filtered_df.show()  # Action

Transformation created:
== Physical Plan ==
*(1) Filter (isnotnull(amount#1461) AND (amount#1461 > 1000))
+- FileScan csv [city#1458,category#1459,product#1460,amount#1461] Batched: false, DataFilters: [isnotnull(amount#1461), (amount#1461 > 1000)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/sales_week7_lab.csv], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,1000)], ReadSchema: struct<city:string,category:string,product:string,amount:int>


Action result:
+---------+-----------+--------+------+
|     city|   category| product|amount|
+---------+-----------+--------+------+
|  Bangkok|Electronics|  Laptop| 35000|
|ChiangMai|Electronics|  Laptop| 34000|
|  Bangkok|Accessories|Keyboard|  1500|
+---------+-----------+--------+------+



## **Question 4 — Why does Spark use lazy evaluation?**

In [175]:
step1 = csv_df.filter(F.col("amount") > 1000)
step2 = step1.select("city", "amount")

print("Transformations have been defined.")
print("No result has been displayed yet.")

# This action triggers execution
step2.show()

Transformations have been defined.
No result has been displayed yet.
+---------+------+
|     city|amount|
+---------+------+
|  Bangkok| 35000|
|ChiangMai| 34000|
|  Bangkok|  1500|
+---------+------+



Spark uses lazy evaluation so it can wait until an action is requested and then optimize the entire sequence of transformations. This can reduce unnecessary work and improve performance.

## **Question 5 — What does reduceByKey() hide compared with the Week 6 MapReduce simulation?**

In [176]:
sales_rdd = spark.sparkContext.parallelize(
    [(row["city"], row["amount"]) for row in csv_df.collect()]
)

city_totals = sales_rdd.reduceByKey(
    lambda a, b: a + b
)

print("City totals using reduceByKey():")
print(city_totals.collect())

City totals using reduceByKey():
[('Bangkok', 37000), ('ChiangMai', 34100), ('Phuket', 120)]


reduceByKey() hides much of the manual MapReduce process. Instead of manually grouping values by key and then reducing them, Spark automatically performs the key-based aggregation.

## **Question 6 — Why can groupBy() lead to a shuffle?**

In [177]:
grouped_df = csv_df.groupBy("city").sum("amount")

print("Execution plan for groupBy():")
grouped_df.explain()

grouped_df.show()

Execution plan for groupBy():
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#1458], functions=[sum(amount#1461)])
   +- Exchange hashpartitioning(city#1458, 200), ENSURE_REQUIREMENTS, [plan_id=3682]
      +- HashAggregate(keys=[city#1458], functions=[partial_sum(amount#1461)])
         +- FileScan csv [city#1458,amount#1461] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/sales_week7_lab.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string,amount:int>


+---------+-----------+
|     city|sum(amount)|
+---------+-----------+
|   Phuket|        120|
|ChiangMai|      34100|
|  Bangkok|      37000|
+---------+-----------+



groupBy() can cause a shuffle because Spark needs to bring records with the same grouping key together. If those records are located in different partitions, Spark may need to move data between partitions.

## **Question 7 — Why can collect() be dangerous on Big Data?**

In [178]:
print("Number of records:", csv_df.count())

# Safe for this very small Part J dataset:
data = csv_df.collect()

print("Collected records:")
print(data)

Number of records: 6
Collected records:
[Row(city='Bangkok', category='Electronics', product='Laptop', amount=35000), Row(city='Bangkok', category='Accessories', product='Mouse', amount=500), Row(city='ChiangMai', category='Electronics', product='Laptop', amount=34000), Row(city='Bangkok', category='Accessories', product='Keyboard', amount=1500), Row(city='Phuket', category='Food', product='Coffee', amount=120), Row(city='ChiangMai', category='Food', product='Coffee', amount=100)]


collect() brings all data into the driver's memory. It is safe for the small Part J dataset, but it can be dangerous with Big Data because millions or billions of records could exceed the driver's available memory.

## **Question 8 — When would you prefer a DataFrame over an RDD for structured analytics?**

In [179]:
result = (
    csv_df
    .groupBy("category")
    .agg(
        F.sum("amount").alias("total_sales"),
        F.avg("amount").alias("average_sales"),
        F.count("*").alias("transactions")
    )
    .orderBy(F.desc("total_sales"))
)

result.show()

+-----------+-----------+-------------+------------+
|   category|total_sales|average_sales|transactions|
+-----------+-----------+-------------+------------+
|Electronics|      69000|      34500.0|           2|
|Accessories|       2000|       1000.0|           2|
|       Food|        220|        110.0|           2|
+-----------+-----------+-------------+------------+



I would prefer a DataFrame for structured analytics because it has named columns and supports convenient operations such as filter(), groupBy(), sum(), avg(), and count(). Spark can also optimize DataFrame queries.

## **Question 9 — What is the advantage of validating the same totals with both RDD and DataFrame code?**

In [180]:
rdd_totals = dict(
    spark.sparkContext.parallelize(
        [(row["city"], row["amount"]) for row in csv_df.collect()]
    )
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# Calculate city totals using DataFrame
df_totals = {
    row["city"]: row["total_sales"]
    for row in (
        csv_df.groupBy("city")
        .agg(F.sum("amount").alias("total_sales"))
        .collect()
    )
}

print("RDD totals:", rdd_totals)
print("DataFrame totals:", df_totals)
print("Same totals?", rdd_totals == df_totals)

RDD totals: {'Bangkok': 37000, 'ChiangMai': 34100, 'Phuket': 120}
DataFrame totals: {'Phuket': 120, 'ChiangMai': 34100, 'Bangkok': 37000}
Same totals? True


Using both RDD and DataFrame code to calculate the same totals provides a way to cross-check the results. If both approaches produce the same totals, it increases confidence that the calculations are correct.

In [181]:
spark.stop()